# ViGoEmotions Dataset Validation

## HAN-18: Dataset Verification

This notebook validates the ViGoEmotions dataset.

Dataset: 20,664 Vietnamese comments
Task: Multi-label emotion classification

## Kiểm tra môi trường
Chọn kernel **Python (ViGoEmotions)** (`ViGoEmotions/.venv/bin/python`), rồi **Restart Kernel → Run All**. Cell dưới kiểm tra dependency trước khi xử lý dữ liệu.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from check_environment import check_environment
check_environment()

Python kernel: /Users/mac/Documents/Studyspace/2026-2027/Semester_1/ATI/ViGoEmotions/.venv/bin/python
numpy: 2.5.4
scipy: 1.18.1
scikit-learn: 1.9.1


NumPy / SciPy / scikit-learn imports: OK


In [2]:
import pandas as pd
from pathlib import Path

# Hỗ trợ chạy Notebook từ project root
# hoặc từ folder notebooks
data_dir = Path("data")
if not data_dir.exists():
    data_dir = Path("../data")

train_df = pd.read_csv(data_dir / "train.csv")
val_df = pd.read_csv(data_dir / "val.csv")
test_df = pd.read_csv(data_dir / "test.csv")

datasets = {
    "train": train_df,
    "val": val_df,
    "test": test_df
}

full_df = pd.concat(
    datasets.values(),
    ignore_index=True
)

In [3]:
summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test", "Total"],
    "Samples": [
        len(train_df),
        len(val_df),
        len(test_df),
        len(full_df)
    ]
})

display(summary)

,Split,Samples
0,Train,16531
1,Validation,2066
2,Test,2067
3,Total,20664


In [4]:
print("Missing values:")
display(full_df.isnull().sum())

print("Duplicate IDs:",
      full_df.duplicated(subset=["id"]).sum())

print("Duplicate texts:",
      full_df.duplicated(subset=["text"]).sum())

print("Dataset schema:")
display(full_df.dtypes)

Missing values:


id        0
text      0
labels    0
dtype: int64

Duplicate IDs: 0
Duplicate texts: 1
Dataset schema:


id        str
text      str
labels    str
dtype: object

In [5]:
parts = []

for split, df in datasets.items():
    temp = df.copy()
    temp["split"] = split
    parts.append(temp)

combined = pd.concat(parts, ignore_index=True)

duplicates = combined[
    combined.duplicated(subset=["text"], keep=False)
]

print(duplicates[["id", "split", "labels"]].to_string(index=False))

cross_split = (
    duplicates.groupby("text")["split"]
    .nunique()
    .gt(1)
    .sum()
)

print("Texts duplicated across splits:", cross_split)


       id split labels
     1698 train   [11]
red000094  test   [11]
Texts duplicated across splits: 1


## HAN-34: Dataset Schema Inspection

### Objective
Inspect the structure of the ViGoEmotions dataset and
document the meaning, type, and format of each column.

### Dataset
- Total samples: 20,664
- Train: 16,531
- Validation: 2,066
- Test: 2,067

In [6]:

print("=== DATASET SCHEMA ===")

schema = pd.DataFrame({
    "Column": full_df.columns,
    "Data Type": full_df.dtypes.astype(str).values,
    "Non-null Count": full_df.notna().sum().values,
    "Unique Values": full_df.nunique().values,
})

display(schema)


=== DATASET SCHEMA ===


,Column,Data Type,Non-null Count,Unique Values
0,id,str,20664,20664
1,text,str,20664,20663
2,labels,str,20664,2549


In [7]:

data_dictionary = pd.DataFrame([
    {
        "Column": "id",
        "Type": "str",
        "Description": "Unique identifier for a comment"
    },
    {
        "Column": "text",
        "Type": "str",
        "Description": "Vietnamese social-media comment"
    },
    {
        "Column": "labels",
        "Type": "str",
        "Description": "Serialized emotion label indices"
    }
])

display(data_dictionary)


,Column,Type,Description
0,id,str,Unique identifier for a comment
1,text,str,Vietnamese social-media comment
2,labels,str,Serialized emotion label indices


In [8]:

LABEL_NAMES = [
    "amusement", "excitement", "joy", "love",
    "desire", "optimism", "caring", "pride",
    "admiration", "gratitude", "relief", "approval",
    "realization", "surprise", "curiosity",
    "confusion", "fear", "nervousness", "remorse",
    "embarrassment", "disappointment", "sadness",
    "grief", "disgust", "anger", "annoyance",
    "disapproval", "neutral"
]

LABEL_MAP = dict(enumerate(LABEL_NAMES))

label_table = pd.DataFrame([
    {"Index": idx, "Emotion": name}
    for idx, name in LABEL_MAP.items()
])

display(label_table)


,Index,Emotion
0,0,amusement
1,1,excitement
2,2,joy
3,3,love
4,4,desire
5,5,optimism
6,6,caring
7,7,pride
8,8,admiration
9,9,gratitude


In [9]:

import ast

def parse_labels(value):
    if isinstance(value, str):
        parsed = ast.literal_eval(value)
    else:
        parsed = value

    if not isinstance(parsed, list):
        raise ValueError("Labels must be a list")

    if not all(type(x) is int for x in parsed):
        raise ValueError("Label IDs must be integers")

    return parsed

full_df = full_df.copy()

parsed_labels = []
errors = []

for idx, value in enumerate(full_df["labels"]):
    try:
        parsed_labels.append(parse_labels(value))
    except (ValueError, SyntaxError, TypeError) as exc:
        errors.append((idx, str(exc)))

print("Parsing errors:", len(errors))

if errors:
    print("First 5 errors:", errors[:5])
    raise ValueError("Fix label parsing before continuing")

full_df["label_ids"] = parsed_labels

display(full_df[["id", "labels", "label_ids"]].head(10))


Parsing errors: 0


,id,labels,label_ids
0,tik000008,[12],[12]
1,5743,"[2, 8, 3]","[2, 8, 3]"
2,32895,"[8, 7]","[8, 7]"
3,you001182,[27],[27]
4,12052,"[24, 23]","[24, 23]"
5,1824,"[12, 13]","[12, 13]"
6,tik023326,[7],[7]
7,1046,[0],[0]
8,11556,[27],[27]
9,2320,"[0, 11, 2, 3]","[0, 11, 2, 3]"


In [10]:

all_label_ids = sorted({
    label
    for labels in full_df["label_ids"]
    for label in labels
})

invalid_ids = sorted(
    set(all_label_ids) - set(range(28))
)

duplicate_label_rows = full_df["label_ids"].apply(
    lambda labels: len(labels) != len(set(labels))
).sum()

neutral_mixed_rows = full_df["label_ids"].apply(
    lambda labels: 27 in labels and len(set(labels)) > 1
).sum()

neutral_only_rows = full_df["label_ids"].apply(
    lambda labels: labels == [27]
).sum()

empty_label_rows = full_df["label_ids"].apply(
    lambda labels: len(labels) == 0
).sum()

emotion_ids = [i for i in all_label_ids if i != 27]

print("=== LABEL VALIDATION ===")
print("Observed label IDs:", all_label_ids)
print("Emotion categories observed:", len(emotion_ids))
print("Invalid IDs:", invalid_ids)
print("Rows with repeated labels:", duplicate_label_rows)
print("Neutral-only rows:", neutral_only_rows)
print("Neutral mixed with emotions:", neutral_mixed_rows)
print("Empty-label rows:", empty_label_rows)


=== LABEL VALIDATION ===
Observed label IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Emotion categories observed: 27
Invalid IDs: []
Rows with repeated labels: 1
Neutral-only rows: 725
Neutral mixed with emotions: 306
Empty-label rows: 0


In [11]:

import numpy as np

def to_multihot(labels, num_classes=27):
    vector = np.zeros(num_classes, dtype=int)

    # Neutral is represented by an all-zero
    # emotion vector in this modeling convention.
    if labels == [27]:
        return vector

    for label in set(labels) - {27}:
        if not 0 <= label < num_classes:
            raise ValueError(f"Invalid emotion label: {label}")
        vector[label] = 1

    return vector

# Demo
print("Labels [2, 11]:")
print(to_multihot([2, 11]))

print("\nNeutral [27]:")
print(to_multihot([27]))


Labels [2, 11]:
[0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Neutral [27]:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


## Quy ước thống nhất cho midterm
Có 306 mẫu neutral + cảm xúc: bỏ ID 27 khi mã hóa, giữ các cảm xúc còn lại. Neutral riêng lẻ là vector 0; nhãn lặp được gộp. Loại train ID 1698 do trùng văn bản test. Xem `scripts/common.py` và `reports/midterm_report.md`. Không xuất comment thô trong notebook công khai.